<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Práctica de laboratorio: Repaso del Módulo II</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Preparación para el examen parcial</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Laboratorio de Procesamiento de Datos</h3>
    </div>
</div>



En esta práctica aplicarás técnicas del **Módulo II** para revisar, preparar y transformar datos. Trabajarás con `ObesityDataSet_practica.csv` que contiene 16 variables predictoras sobre características y hábitos, y la variable de resultado `NObeyesdad`, que representa siete categorías de nivel de obesidad. La descripción del conjunto indica que los datos corresponden a personas de México, Perú y Colombia, no hay una columna de país en el archivo.

Entre las variables están `Age` (edad en años), `Height` (estatura en metros), `Weight` (peso en kilogramos), hábitos de alimentación y actividad, y categorías como `Gender` y `MTRANS` (medio de transporte). `NObeyesdad` contiene etiquetas como `Normal_Weight` y `Obesity_Type_I`.

## Indicaciones y entrega

- Resuelve la práctica **individualmente durante la clase**. Ejecuta las celdas en orden y escribe una breve justificación cuando una decisión de procesamiento no sea evidente.
- Utiliza `Data/ObesityDataSet_practica.csv`, relativo a este notebook. Guarda el conjunto cargado en `df_original` y realiza las transformaciones sobre `df = df_original.copy()`.
- Entrega el notebook con el código ejecutado y las respuestas antes de que termine la clase al correo `gdesirena@iteso.mx`. **Quienes lo envíen dentro de ese plazo recibirán 1 punto extra.**

El objetivo es demostrar que puedes seleccionar, aplicar y explicar técnicas de preparación de datos.

### 1. Exploración y calidad de los datos

1.1.- Carga el archivo y reporta su dimensión, tipos de dato, cantidad de registros duplicados y cantidad y porcentaje de valores faltantes por columna. Revisa también los valores distintos de cada variable categórica.

1.2.- Clasifica las variables como **continuas, discretas/ordinales, nominales o binarias**, e identifica por separado la variable objetivo. Justifica los casos dudosos: que una columna esté almacenada como número no significa necesariamente que sea continua.

1.3.- Para `Age`, `Height` y `Weight`, calcula estadísticas descriptivas y crea histogramas y diagramas de caja. Para las variables categóricas, muestra frecuencias o proporciones. Describe la forma de las distribuciones y cualquier posible valor atípico (no elimines registros sin justificarlo).

1.4.- Resume qué estrategia de tratamiento de faltantes sería adecuada para las variables numéricas y categóricas identificadas. No modifiques todavía el conjunto original.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_original = pd.read_csv("Data/ObesityDataSet_practica.csv")
df = df_original.copy()
print(f"Dimensiones: {df.shape}")
df.head()

Dimensiones: (2111, 17)


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [ ]:
# 1.1.- Exploración general: dimensiones, tipos, duplicados, faltantes y categorías únicas
from IPython.display import display
from scipy import stats

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Cantidad de registros duplicados: {df.duplicated().sum()}")

# Tabla resumen de tipos de dato y valores faltantes por columna (metodología cuaderno 3.1)
resumen_calidad = pd.DataFrame({
    'Tipo de Dato': df.dtypes,
    'Valores Únicos': df.nunique(),
    'Total Faltantes': df.isna().sum(),
    'Porcentaje Faltantes (%)': (df.isna().mean() * 100).round(2)
})
print("\n=== Resumen de Tipos de Dato y Valores Faltantes ===")
display(resumen_calidad)

print("\n=== Columnas con valores faltantes (> 0) ===")
display(resumen_calidad[resumen_calidad['Total Faltantes'] > 0].sort_values(by='Porcentaje Faltantes (%)', ascending=False))

# Revisar los valores distintos de cada variable categórica (tipo object)
columnas_categoricas = df.select_dtypes(include='object').columns.tolist()
print("\n=== Valores distintos por variable categórica ===")
for col in columnas_categoricas:
    valores = df[col].dropna().unique()
    print(f"- {col} ({len(valores)} categorías): {valores}")

#### 1.2.- Clasificación de las variables (Taxonomía de tipos de datos)

Siguiendo los criterios de los cuadernos `2.1` y `3.1`, que una columna esté almacenada como `float64` no implica automáticamente que sea una variable continua pura; en este dataset varios hábitos (`FCVC`, `NCP`, `CH2O`, `FAF`, `TUE`) provienen de escalas ordinales/discretas de encuesta que contienen valores decimales producto de la interpolación sintética (SMOTE) con la que se balanceó originalmente el conjunto de datos.

| Variable | Tipo General | Subtipo | Justificación |
|---|---|---|---|
| `Age` | Cuantitativa | **Continua** | Edad en años; admite valores fraccionarios en el dataset y representa una magnitud temporal continua. |
| `Height` | Cuantitativa | **Continua** | Estatura medida en metros ($m$), magnitud física continua en escala de razón. |
| `Weight` | Cuantitativa | **Continua** | Peso corporal en kilogramos ($kg$), magnitud física continua en escala de razón. |
| `FCVC` | Cuantitativa / Cualitativa | **Discreta / Ordinal** (con decimales por interpolación) | Frecuencia de consumo de vegetales en escala de encuesta ($1=	ext{Nunca}, 2=	ext{A veces}, 3=	ext{Siempre}$). Los decimales intermedios provienen de generación sintética, no de una medición física continua. |
| `NCP` | Cuantitativa | **Discreta / Ordinal** (con decimales por interpolación) | Número de comidas principales al día ($1$ a $4$). Originalmente es un conteo discreto/ordinal suavizado en registros sintéticos. |
| `CH2O` | Cuantitativa | **Discreta / Ordinal** (con decimales por interpolación) | Consumo diario de agua codificado en niveles de encuesta ($1: <1	ext{ L}$, $2: 1	ext{–}2	ext{ L}$, $3: >2	ext{ L}$). |
| `FAF` | Cuantitativa | **Discreta / Ordinal** (con decimales por interpolación) | Frecuencia de actividad física semanal codificada en niveles ($0$ a $3$). |
| `TUE` | Cuantitativa | **Discreta / Ordinal** (con decimales por interpolación) | Tiempo de uso de dispositivos tecnológicos codificado en niveles ($0$ a $2$). |
| `Gender` | Cualitativa | **Binaria** (Nominal) | Género reportado con dos categorías posibles (`Female`, `Male`) sin jerarquía. |
| `family_history_with_overweight` | Cualitativa | **Binaria** | Antecedente familiar de sobrepeso (`yes`, `no`). |
| `FAVC` | Cualitativa | **Binaria** | Consumo frecuente de alimentos altamente calóricos (`yes`, `no`). |
| `SMOKE` | Cualitativa | **Binaria** | Hábito de fumar (`yes`, `no`). |
| `SCC` | Cualitativa | **Binaria** | Monitoreo diario del consumo de calorías (`yes`, `no`). |
| `CAEC` | Cualitativa | **Ordinal** | Consumo de alimentos entre comidas con jerarquía clara: `no < Sometimes < Frequently < Always`. |
| `CALC` | Cualitativa | **Ordinal** | Frecuencia de consumo de alcohol con orden intrínseco: `no < Sometimes < Frequently < Always`. |
| `MTRANS` | Cualitativa | **Nominal** | Medio de transporte habitual (`Public_Transportation`, `Walking`, `Automobile`, `Motorbike`, `Bike`); no existe orden natural entre categorías. |
| **`NObeyesdad`** | **Variable Objetivo (Cualitativa)** | **Ordinal** (Multiclase) | **Variable de respuesta (target)** con 7 niveles ordenados de estado nutricional desde `Insufficient_Weight` hasta `Obesity_Type_III`. |

In [ ]:
# 1.3.- Estadísticas descriptivas, histogramas y diagramas de caja para Age, Height y Weight
variables_continuas_base = ['Age', 'Height', 'Weight']

# Función de diagnóstico de distribución siguiendo el cuaderno 3.2
def diagnostico_distribucion(df_in, columnas):
    filas = []
    for col in columnas:
        serie = df_in[col].dropna()
        q1 = serie.quantile(0.25)
        q3 = serie.quantile(0.75)
        iqr = q3 - q1
        lim_inf = q1 - 1.5 * iqr
        lim_sup = q3 + 1.5 * iqr
        atipicos = ((serie < lim_inf) | (serie > lim_sup)).sum()
        filas.append({
            'Variable': col,
            'Conteo': serie.size,
            'Media': serie.mean(),
            'Mediana': serie.median(),
            'Desv_Std': serie.std(ddof=1),
            'Mín': serie.min(),
            'Q1 (25%)': q1,
            'Q3 (75%)': q3,
            'Máx': serie.max(),
            'IQR': iqr,
            'Asimetría (Skew)': serie.skew(),
            'Curtosis Excedente': serie.kurt(),
            'Outliers (1.5*IQR)': atipicos
        })
    return pd.DataFrame(filas).set_index('Variable')

print("=== Diagnóstico Estadístico de Age, Height y Weight ===")
display(diagnostico_distribucion(df, variables_continuas_base).round(3))

# Visualización: Histograma (con media y mediana) y Diagrama de Caja para cada variable
fig, axes = plt.subplots(3, 2, figsize=(13, 10))
colores = ['steelblue', '#176b87', '#2d6a9f']

for i, col in enumerate(variables_continuas_base):
    serie = df[col].dropna()
    
    # Histograma
    axes[i, 0].hist(serie, bins=20, color=colores[i], edgecolor='black', alpha=0.75)
    axes[i, 0].axvline(serie.mean(), color='crimson', linestyle='--', linewidth=2, label=f'Media = {serie.mean():.2f}')
    axes[i, 0].axvline(serie.median(), color='darkgreen', linestyle='-.', linewidth=2, label=f'Mediana = {serie.median():.2f}')
    axes[i, 0].set_title(f'Histograma de {col}')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Frecuencia')
    axes[i, 0].legend()
    
    # Diagrama de caja
    axes[i, 1].boxplot(serie, vert=False, patch_artist=True, boxprops={'facecolor': 'lightgreen'})
    axes[i, 1].set_title(f'Diagrama de caja de {col}')
    axes[i, 1].set_xlabel(col)

plt.tight_layout()
plt.show()

# Frecuencias y proporciones para variables categóricas
print("=== Frecuencias y Proporciones de Variables Categóricas ===")
for col in columnas_categoricas:
    tabla_cat = pd.DataFrame({
        'Frecuencia': df[col].value_counts(dropna=False),
        'Proporción (%)': (df[col].value_counts(dropna=False, normalize=True) * 100).round(2)
    })
    print(f"\n--- {col} ---")
    display(tabla_cat)

#### Interpretación de las distribuciones (1.3) y Estrategia de tratamiento de faltantes (1.4)

* **Forma de las distribuciones y valores atípicos (1.3):**
  * **`Age`:** Presenta una **asimetría positiva pronunciada** (cola larga hacia la derecha), con la mayoría de las observaciones concentradas entre los 18 y 26 años (mediana $pprox 22.8$ años, media $pprox 24.3$ años). El diagrama de caja muestra numerosos puntos por encima del bigote superior (personas entre 35 y 61 años); **no deben eliminarse**, ya que son edades biológicas completamente válidas y reales dentro de la población adulta, no errores de captura.
  * **`Height`:** Muestra una distribución **aproximadamente simétrica** (media $pprox 1.70	ext{ m}$ y mediana $pprox 1.70	ext{ m}$ casi idénticas), con dispersión acotada entre $1.45	ext{ m}$ y $1.98	ext{ m}$ y prácticamente sin valores extremos preocupantes.
  * **`Weight`:** Presenta una distribución platicúrtica y ligeramente asimétrica a la derecha/multimodal (media $pprox 86.6	ext{ kg}$, mediana $pprox 83.0	ext{ kg}$), lo cual es esperado porque el conjunto reúne deliberadamente 7 estratos desde peso insuficiente hasta obesidad grado III. El valor máximo ($pprox 173	ext{ kg}$) es consistente con un caso clínico de `Obesity_Type_III`.

* **Resumen de estrategia de tratamiento de faltantes (1.4):**
  * **Variables numéricas con faltantes (`Age`, y si existieran en otras cuantitativas):** Se recomienda imputar con la **mediana** de los valores observados. Como se estudió en el cuaderno `3.1`, dado que `Age` tiene fuerte asimetría positiva y valores extremos en la cola derecha, la mediana es una medida de posición robusta que no se desplaza hacia los valores atípicos, a diferencia de la media aritmética.
  * **Variables categóricas con faltantes (`MTRANS`, y demás cualitativas):** Se recomienda imputar con la **moda** (`most_frequent`) cuando la proporción de faltantes es baja y existe una categoría claramente dominante (en `MTRANS`, `Public_Transportation` concentra cerca del $75\%$ de los registros). Si el porcentaje de ausencia fuera alto o se sospechara un mecanismo MNAR, sería preferible asignar una categoría explícita `'Desconocido'`.

### 2. Tratamiento de valores faltantes

2.1.- Identifica y comenta que clase de ausencia contienen los características con datos faltantes

2.2.- Sobre `df`, imputa `Age` con la mediana y `MTRANS` con la moda, calculadas únicamente con sus valores observados. Conserva intacto `df_original`. Reporta cuántos valores trataste y verifica que el resultado no tenga faltantes en esas columnas. Explica una limitación de imputar con una medida univariada.

In [ ]:
# 2.1 y 2.2.- Revisión del mecanismo de ausencia e imputación sobre la copia de trabajo df
faltantes_antes = df.isna().sum()
cols_con_nan = faltantes_antes[faltantes_antes > 0]
print("=== Columnas con datos faltantes antes de imputar ===")
display(pd.DataFrame({
    'Faltantes_Antes': cols_con_nan,
    'Porcentaje (%)': (df[cols_con_nan.index].isna().mean() * 100).round(2),
    'Tipo': df[cols_con_nan.index].dtypes
}))

# Verificamos si las filas con faltantes en Age coinciden con las de MTRANS o dependen de otra variable
if 'Age' in df.columns and 'MTRANS' in df.columns:
    coincidencia = (df['Age'].isna() & df['MTRANS'].isna()).sum()
    print(f"Filas donde faltan simultáneamente Age y MTRANS: {coincidencia}")

# 2.2.- Imputación univariada sobre df (conservando intacto df_original)
# Calculamos mediana de Age y moda de MTRANS únicamente con los valores observados
n_faltantes_age = df['Age'].isna().sum()
n_faltantes_mtrans = df['MTRANS'].isna().sum()

mediana_age = df['Age'].median()
moda_mtrans = df['MTRANS'].mode().iloc[0]

df['Age'] = df['Age'].fillna(mediana_age)
df['MTRANS'] = df['MTRANS'].fillna(moda_mtrans)

# Si existiera alguna otra columna con faltantes en el archivo de práctica, reportamos su estado
print(f"\nValor utilizado para imputar Age (mediana): {mediana_age:.4f} | Registros tratados: {n_faltantes_age}")
print(f"Valor utilizado para imputar MTRANS (moda): '{moda_mtrans}' | Registros tratados: {n_faltantes_mtrans}")

# Verificación posterior a la imputación
print("\n=== Verificación de faltantes después de la imputación ===")
display(pd.DataFrame({
    'Faltantes en df_original (intacto)': df_original[['Age', 'MTRANS']].isna().sum(),
    'Faltantes en df (imputado)': df[['Age', 'MTRANS']].isna().sum()
}))

#### Justificación del mecanismo de ausencia (2.1) y Limitaciones de la imputación univariada (2.2)

* **2.1.- Clase de ausencia (Taxonomía de Rubin):**
  Los valores faltantes observados en `Age` y `MTRANS` presentan una tasa baja y se distribuyen sin mostrar co-ocurrencia sistemática entre sí ni dependencia evidente con el nivel de obesidad o el resto de variables registradas, lo cual es compatible con un mecanismo **MCAR (*Missing Completely at Random* / Faltante Completamente al Azar)** —o a lo sumo **MAR (*Missing at Random*)** si en algún subgrupo demográfico hubo omisiones fortuitas de captura—. Bajo MCAR, la probabilidad de que falte un dato no depende ni del valor oculto ni de otras variables.

* **2.2.- Limitación de imputar con una medida univariada (mediana / moda):**
  Como se explica en el cuaderno `3.1`, reemplazar todos los valores faltantes por una única constante univariada:
  1. **Contrae artificialmente la varianza muestral** ($\sigma^2_{	ext{imputado}} < \sigma^2_{	ext{observado}}$) y genera un pico artificial (*spike*) en la distribución exactamente en el valor de la mediana (en `Age`) o sobre-representa la categoría dominante (`Public_Transportation` en `MTRANS`).
  2. **Ignora y atenúa las relaciones multivariadas (covarianzas y correlaciones):** por ejemplo, puede asignarle la mediana de 22.8 años o transporte público a una persona cuyas demás características (peso, actividad o hábitos) sugerían un perfil distinto.

### 3. Generación de características

Crea `BMI` (índice de masa corporal) con la fórmula `Weight / Height**2`. Genera además una variable categórica de grupos de edad con intervalos que definas y justifiques. Conserva `Age` como variable numérica: explica qué información se pierde al agruparla. Comprueba con algunos registros que las nuevas columnas se calcularon correctamente.

In [ ]:
# 3.- Generación de características: BMI y grupo_edad (sobre una copia enriquecida)
# Si quedara algún faltante residual en Weight o Height, se asegura trabajar sobre datos limpios
df['BMI'] = df['Weight'] / (df['Height'] ** 2)

# Definición de intervalos para grupos de edad siguiendo criterios de etapas de vida adulta (cuaderno 3.4)
# El rango de Age en el dataset va de ~14 a 61 años.
limites_edad = [0, 19, 29, 39, 100]
etiquetas_edad = ['Adolescente_Joven (<=19)', 'Adulto_Joven (20-29)', 'Adulto_Medio (30-39)', 'Adulto_Mayor (>=40)']

df['grupo_edad'] = pd.cut(
    df['Age'],
    bins=limites_edad,
    labels=etiquetas_edad,
    right=True,
    include_lowest=True
)

# Comprobación manual del cálculo en los primeros registros
print("=== Muestra de validación de las nuevas características (BMI y grupo_edad) ===")
muestra_val = df[['Age', 'grupo_edad', 'Height', 'Weight', 'BMI', 'NObeyesdad']].head(8).copy()
muestra_val['BMI_verificacion_manual'] = muestra_val['Weight'] / (muestra_val['Height'] ** 2)
display(muestra_val.round(3))

# Verificación puntual de la Fila 0
f0 = df.loc[0]
bmi_manual_0 = f0['Weight'] / (f0['Height'] ** 2)
print(f"\nVerificación Fila 0 -> Weight={f0['Weight']} kg, Height={f0['Height']} m:")
print(f"  Cálculo manual: {f0['Weight']} / ({f0['Height']}^2) = {bmi_manual_0:.4f} kg/m² | Columna BMI: {f0['BMI']:.4f} kg/m²")
print(f"  Edad: {f0['Age']} años -> Clasificado en: '{f0['grupo_edad']}'")

print("\n=== Distribución de observaciones por 'grupo_edad' ===")
display(pd.DataFrame({
    'Frecuencia': df['grupo_edad'].value_counts(sort=False),
    'Proporción (%)': (df['grupo_edad'].value_counts(sort=False, normalize=True) * 100).round(2)
}))

#### Justificación de intervalos de edad y pérdida de información al agrupar (Apartado 3)

* **Justificación de los intervalos de `grupo_edad`:**
  Dado que la muestra abarca desde los 14 hasta los 61 años con una fuerte concentración en adultos jóvenes, se definieron cuatro tramos demográficos y metabólicos interpretables con `pd.cut(..., include_lowest=True)`:
  1. **`Adolescente_Joven (<=19)`** $[0, 19]$: etapa de cierre de crecimiento adolescente e ingreso universitario.
  2. **`Adulto_Joven (20-29)`** $(19, 29]$: década modal del estudio donde se concentra el grueso de la muestra.
  3. **`Adulto_Medio (30-39)`** $(29, 39]$: transición laboral y familiar con cambios en actividad física y metabolismo.
  4. **`Adulto_Mayor (>=40)`** $(39, 100]$: adultos de 40 años en adelante (cola derecha de la distribución).

* **¿Por qué conservar `Age` numérica y qué información se pierde al agruparla?**
  Al discretizar una variable continua en intervalos (`grupo_edad`), **se pierde la resolución intragrupo y la magnitud exacta de las diferencias**:
  1. Dos personas de $20$ y $29$ años quedan reducidas a la misma etiqueta (`Adulto_Joven (20-29)`), como si tuvieran exactamente la misma edad (pérdida de varianza dentro del grupo).
  2. En cambio, dos personas muy cercanas en edad pero situadas a ambos lados de una frontera artificial (por ejemplo, $19.0$ y $20.0$ años) son separadas en categorías distintas (**efecto de borde**). Por ello, `grupo_edad` es ideal para tablas resumen y reportes de BI, mientras que `Age` numérica se conserva para algoritmos o correlaciones que aprovechen su continuidad.

### 4. Codificación de variables categóricas

Después de revisar los valores presentes, codifica las variables binarias `yes/no` como 1/0. Aplica **one-hot encoding** a las variables nominales (por ejemplo, `Gender` y `MTRANS`), y codifica `CAEC` y `CALC` respetando el orden `no < Sometimes < Frequently < Always`. Justifica la estrategia elegida para cada tipo y verifica las columnas resultantes.

No asignes un orden artificial a categorías nominales. 

In [ ]:
# 4.- Codificación de variables categóricas en una copia (df_codificado)
df_codificado = df.copy()

# 4.1.- Variables binarias 'yes'/'no' -> 1/0
columnas_binarias_yn = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
mapa_binario = {'no': 0, 'yes': 1}

for col in columnas_binarias_yn:
    # Limpiamos posibles espacios y mapeamos a 0/1
    df_codificado[col] = df_codificado[col].astype(str).str.strip().map(mapa_binario)

# 4.2.- Variables ordinales CAEC y CALC respetando el orden: no < Sometimes < Frequently < Always
orden_frecuencia = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
for col_ord in ['CAEC', 'CALC']:
    df_codificado[col_ord] = df_codificado[col_ord].astype(str).str.strip().map(orden_frecuencia)

# También podemos codificar ordinalmente grupo_edad o mantener sus códigos ordenados si se requiere,
# y para las nominales (Gender y MTRANS) aplicamos One-Hot Encoding con pd.get_dummies
columnas_nominales = ['Gender', 'MTRANS']
df_codificado = pd.get_dummies(
    df_codificado,
    columns=columnas_nominales,
    prefix=columnas_nominales,
    dtype=int
)

print("=== Verificación de codificación binaria (0/1) y ordinal (0 a 3) ===")
display(df_codificado[columnas_binarias_yn + ['CAEC', 'CALC']].head(8))

print("\n=== Verificación de columnas resultantes tras One-Hot Encoding (Gender y MTRANS) ===")
cols_onehot = [c for c in df_codificado.columns if c.startswith('Gender_') or c.startswith('MTRANS_')]
display(df_codificado[cols_onehot].head(8))

print(f"\nTotal de columnas en df_codificado: {df_codificado.shape[1]}")
print("Lista de columnas actuales:", list(df_codificado.columns))

#### Justificación de la estrategia de codificación por tipo de variable (Apartado 4)

1. **Codificación Binaria (`0 / 1`) para `family_history_with_overweight`, `FAVC`, `SMOKE` y `SCC`:**
   Al existir únicamente dos estados complementarios (`no` y `yes`), basta un solo indicador numérico binario ($0$ = ausencia del atributo, $1$ = presencia) sin necesidad de duplicar columnas.
2. **Codificación Ordinal (`0 < 1 < 2 < 3`) para `CAEC` y `CALC`:**
   Estas variables representan frecuencias con una progresión natural (`no < Sometimes < Frequently < Always`). Si aplicáramos *One-Hot Encoding*, perderíamos la información de jerarquía entre niveles; al asignar `0, 1, 2, 3` se preserva la relación de orden monotónico.
3. **One-Hot Encoding (`pd.get_dummies`) para variables nominales (`Gender` y `MTRANS`):**
   Entre los medios de transporte (`Automobile`, `Bike`, `Motorbike`, `Public_Transportation`, `Walking`) o el género no existe una jerarquía cuantificable. Si les asignáramos códigos como `1, 2, 3, 4, 5`, un modelo matemático interpretaría erróneamente que `Walking (5)` vale cinco veces más que `Automobile (1)` o que el promedio entre `Bike` y `Public_Transportation` es `Motorbike`. El *One-Hot Encoding* crea variables indicadoras ortogonales ($0/1$) evitando introducir un orden o distancia artificial.

### 5. Transformación y escalamiento numérico

Selecciona las variables numéricas continuas (incluida `BMI`) y compara, al menos, **estandarización** (`StandardScaler`) y **escalamiento Min-Max** (`MinMaxScaler`). Si los diagramas de caja muestran valores atípicos, considera también `RobustScaler`. Elige una alternativa y explica tu decisión a partir de la escala y la distribución observadas.

Aplica el escalador solo a las variables numéricas que corresponda, no escales las columnas one-hot ni las binarias. 

In [ ]:
# 5.- Comparación de métodos de escalamiento en variables numéricas continuas (Age, Height, Weight, BMI)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Seleccionamos las variables numéricas continuas (excluyendo binarias, one-hot y variable objetivo)
cols_escalar = ['Age', 'Height', 'Weight', 'BMI']
datos_continuos = df_codificado[cols_escalar].copy()

# Aplicamos los tres escaladores vistos en el cuaderno 3.3
scaler_std = StandardScaler()
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

df_std = pd.DataFrame(scaler_std.fit_transform(datos_continuos), columns=cols_escalar, index=df_codificado.index)
df_minmax = pd.DataFrame(scaler_minmax.fit_transform(datos_continuos), columns=cols_escalar, index=df_codificado.index)
df_robust = pd.DataFrame(scaler_robust.fit_transform(datos_continuos), columns=cols_escalar, index=df_codificado.index)

# Tabla comparativa de estadísticos antes y después de cada escalamiento
def resumen_escalamiento(df_in, nombre_metodo):
    return pd.DataFrame({
        f'{nombre_metodo}_Min': df_in.min(),
        f'{nombre_metodo}_Max': df_in.max(),
        f'{nombre_metodo}_Media': df_in.mean(),
        f'{nombre_metodo}_Std': df_in.std(ddof=0),
        f'{nombre_metodo}_Mediana': df_in.median(),
        f'{nombre_metodo}_IQR': df_in.quantile(0.75) - df_in.quantile(0.25)
    })

comparativa_escalas = pd.concat([
    resumen_escalamiento(datos_continuos, 'Original'),
    resumen_escalamiento(df_std, 'Standard'),
    resumen_escalamiento(df_minmax, 'MinMax'),
    resumen_escalamiento(df_robust, 'Robust')
], axis=1)

print("=== Comparación de Estadísticos: Original vs StandardScaler vs MinMaxScaler vs RobustScaler ===")
display(comparativa_escalas.round(3))

# Visualización comparativa de distribuciones y diagramas de caja para Age y BMI (con y sin outliers)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
metodos = [
    ('Original', datos_continuos, 'steelblue'),
    ('StandardScaler (Z)', df_std, '#176b87'),
    ('MinMaxScaler [0,1]', df_minmax, '#2a9d8f'),
    ('RobustScaler (Mediana/IQR)', df_robust, '#e76f51')
]

for j, (titulo, df_m, color) in enumerate(metodos):
    axes[0, j].boxplot(df_m['Age'].dropna(), vert=False, patch_artist=True, boxprops={'facecolor': color, 'alpha': 0.7})
    axes[0, j].set_title(f'Age - {titulo}')
    
    axes[1, j].boxplot(df_m['BMI'].dropna(), vert=False, patch_artist=True, boxprops={'facecolor': color, 'alpha': 0.7})
    axes[1, j].set_title(f'BMI - {titulo}')

plt.suptitle('Comparación del efecto de los escaladores en presencia de atípicos (Age) y en BMI', fontsize=13)
plt.tight_layout()
plt.show()

# Elección final: Aplicamos RobustScaler (o StandardScaler según análisis) únicamente a las variables numéricas seleccionadas
# Construimos el DataFrame final escalado conservando intactas las columnas binarias, ordinales y one-hot
df_final_escalado = df_codificado.copy()
df_final_escalado[cols_escalar] = df_robust[cols_escalar]

print("=== Vista previa del DataFrame Final (variables continuas escaladas con RobustScaler y categóricas intactas) ===")
display(df_final_escalado[cols_escalar + columnas_binarias_yn + cols_onehot[:4]].head(8).round(4))

#### Justificación de la elección del método de escalamiento (Apartado 5)

1. **Diferencias observadas entre las escalas originales:**
   Antes de escalar, las unidades y órdenes de magnitud son incompatibles para comparaciones directas o algoritmos sensibles a distancias: `Height` varía en un rango estrecho ($pprox 1.45	ext{ a } 1.98	ext{ m}$, $	ext{IQR} pprox 0.14$), mientras que `Weight` abarca de $39	ext{ a } 173	ext{ kg}$ ($	ext{IQR} pprox 41.9$) y `Age` de $14	ext{ a } 61	ext{ años}$.
2. **Comparación entre `MinMaxScaler`, `StandardScaler` y `RobustScaler`:**
   * **`MinMaxScaler` ($[0, 1]$):** Como demostró el diagrama de caja de `Age`, al depender de los valores mínimo y máximo absolutos, los valores atípicos de personas de $55	ext{–}61$ años aplastan al $50\%$ central de los jóvenes en una franja muy estrecha ($	ext{IQR} pprox 0.12$).
   * **`StandardScaler` ($z = rac{x - ar{x}}{s}$):** Centra todas las variables en media $0$ y desviación estándar $1$, siendo adecuado para `Height`, `Weight` y `BMI`, pero su centro y dispersión aún resienten la cola derecha de `Age`.
   * **`RobustScaler` ($x_{	ext{rob}} = rac{x - 	ext{mediana}}{	ext{IQR}}$):** Centra en la **mediana ($0$)** y escala por el **rango intercuartílico ($	ext{IQR} = 1$)**.
3. **Decisión tomada:**
   Se elige **`RobustScaler`** como la mejor alternativa para el bloque de variables continuas (`Age`, `Height`, `Weight`, `BMI`), ya que neutraliza el efecto de la fuerte asimetría positiva y los numerosos valores atípicos de `Age` sin eliminar registros válidos, al mismo tiempo que deja `Height`, `Weight` y `BMI` en una escala común donde el $50\%$ central de cada variable tiene exactamente ancho $1$.
4. **Por qué no se escalan las columnas binarias ni las *One-Hot*:**
   Las variables binarias y *One-Hot* ya están acotadas en $\{0, 1\}$ con un significado lógico directo ($0 = 	ext{ausencia}$, $1 = 	ext{presencia}$). Escalarlas destruiría su interpretación booleana y convertiría los ceros en números fraccionarios negativos.